# 项目 · SQL 零售经营分析（参考答案）

> **先独立完成，卡住再看。** 这份答案是给你对照、查漏补缺用的，不是让你直接抄。
> 建议：先自己写，跑通后再对比这里的写法——尤其关注窗口函数和 CTE 的组织方式。
> 本答案全部基于 `data/shop.db`，可直接运行。

---

In [1]:
import sqlite3
import pandas as pd

SHOP_DB = r"C:\Users\千禧\Doubao\chats\2026-09-23\new-chat\AI学习\06_SQL\data\shop.db"
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 60)
conn = sqlite3.connect(SHOP_DB)
def q(sql):
    return pd.read_sql_query(sql, conn)
print("已连接 shop.db")

已连接 shop.db


## 模块 1 · 销售总览

In [2]:
q('''
SELECT COUNT(DISTINCT o.order_id)  AS 订单数,
       COUNT(DISTINCT o.customer_id) AS 客户数,
       ROUND(SUM(oi.sales),0)  AS 总销售额,
       ROUND(SUM(oi.profit),0) AS 总利润,
       ROUND(SUM(oi.sales)/COUNT(DISTINCT o.order_id), 1) AS 客单价,
       ROUND(SUM(oi.profit)*100.0/SUM(oi.sales), 1) AS 利润率_pct
FROM orders o
JOIN order_items oi ON oi.order_id = o.order_id
''')

,订单数,客户数,总销售额,总利润,客单价,利润率_pct
0,2600,300,25040327.0,8233307.0,9630.9,32.9


**结论 1（发现+数字+建议）**：公司 4 年累计销售额约 2504 万、利润 823 万，整体利润率 32.9%，健康。客单价约 9631 元/单（=2504万/2600单）。注意客单价必须用“订单数 2600”作分母，若误用“明细行数 6407”会把客单价算低很多。

## 模块 2 · 品类与地区结构

In [3]:
# 2.1 品类：销售额、利润、利润率
q('''
SELECT p.category,
       ROUND(SUM(oi.sales),0)  AS 销售额,
       ROUND(SUM(oi.profit),0) AS 利润,
       ROUND(SUM(oi.profit)*100.0/SUM(oi.sales),1) AS 利润率_pct
FROM order_items oi
JOIN products p ON p.product_id = oi.product_id
GROUP BY p.category
ORDER BY 销售额 DESC
''')

,category,销售额,利润,利润率_pct
0,科技产品,17257783.0,5774663.0,33.5
1,家具,7386076.0,2346428.0,31.8
2,办公用品,396468.0,112216.0,28.3


In [4]:
# 2.2 地区结构 + 2.3 占比（窗口函数）
q('''
SELECT o.region,
       ROUND(SUM(oi.sales),0) AS 销售额,
       ROUND(SUM(oi.sales)*100.0 / SUM(SUM(oi.sales)) OVER (), 1) AS 占全公司_pct
FROM orders o
JOIN order_items oi ON oi.order_id = o.order_id
GROUP BY o.region
ORDER BY 销售额 DESC
''')

,region,销售额,占全公司_pct
0,华东,6090607.0,24.3
1,华南,5873533.0,23.5
2,华中,5364461.0,21.4
3,西南,4051806.0,16.2
4,华北,3659921.0,14.6


**结论 2**：科技产品是绝对主力（1726 万，占 69%，利润率 33.5%）；家具 739 万；办公用品仅 39.6 万（占 1.6%）。办公用品体量太小，建议评估是否精简 SKU 或与科技产品捆绑。地区上各区域较均衡，华南/华中略高。

## 模块 3 · 客户复购与 RFM

In [5]:
# 3.1 R/F/M：以最新订单日为“今天”
q('''
WITH rfm AS (
  SELECT o.customer_id,
         MAX(o.order_date) AS 最近下单,
         ROUND(julianday((SELECT MAX(order_date) FROM orders))
               - julianday(MAX(o.order_date)), 0) AS recency,
         COUNT(DISTINCT o.order_id) AS frequency,
         ROUND(SUM(oi.sales),0) AS monetary
  FROM orders o
  JOIN order_items oi ON oi.order_id = o.order_id
  GROUP BY o.customer_id
)
SELECT * FROM rfm ORDER BY monetary DESC LIMIT 10
''')

,customer_id,最近下单,recency,frequency,monetary
0,239,2025-12-08,20.0,14,278713.0
1,253,2025-12-09,19.0,13,219833.0
2,160,2025-12-26,2.0,9,213901.0
3,51,2025-12-13,15.0,15,208012.0
4,149,2025-10-17,72.0,15,195392.0
5,123,2025-11-13,45.0,15,192931.0
6,242,2025-06-07,204.0,11,181634.0
7,97,2025-12-23,5.0,11,179549.0
8,166,2025-06-04,207.0,12,179052.0
9,225,2025-10-25,64.0,14,176794.0


In [6]:
# 3.2 按 M 分 4 档（窗口函数不能直接 GROUP BY，要再包一层 CTE）
# 3.3 沉睡客户（recency > 90 天）
q('''
WITH rfm AS (
  SELECT o.customer_id,
         ROUND(julianday((SELECT MAX(order_date) FROM orders))
               - julianday(MAX(o.order_date)), 0) AS recency,
         COUNT(DISTINCT o.order_id) AS frequency,
         SUM(oi.sales) AS monetary
  FROM orders o JOIN order_items oi ON oi.order_id=o.order_id
  GROUP BY o.customer_id
),
ranked AS (
  SELECT *, NTILE(4) OVER (ORDER BY monetary DESC) AS m_band
  FROM rfm
)
SELECT m_band AS M档, COUNT(*) AS 客户数,
       ROUND(AVG(monetary),0) AS 平均消费
FROM ranked GROUP BY m_band ORDER BY m_band
''')

,M档,客户数,平均消费
0,1,75,140261.0
1,2,75,90951.0
2,3,75,65929.0
3,4,75,36730.0


In [7]:
# 沉睡客户数：最近 90 天没下单
q('''
WITH last_order AS (
  SELECT customer_id, MAX(order_date) AS last_d
  FROM orders GROUP BY customer_id
)
SELECT COUNT(*) AS 沉睡客户数
FROM last_order
WHERE julianday((SELECT MAX(order_date) FROM orders)) - julianday(last_d) > 90
''')

,沉睡客户数
0,156


**结论 3**：平均每个客户下单 8.7 次、累计消费约 8.3 万，复购健康。但平均 recency 约 155 天，说明有相当比例客户最近一年没回来——沉睡客户应重点做召回。R 比 M 更该优先运营，因为唤醒老客户比拉新便宜。

## 模块 4 · 员工与组织分析

In [8]:
# 4.1 销售部内部薪资 RANK
q('''
SELECT name, title, region, salary,
       RANK() OVER (ORDER BY salary DESC) AS 薪资排名
FROM employees
WHERE department = '销售部'
ORDER BY 薪资排名
''')

,name,title,region,salary,薪资排名
0,销售总监,区域总监,全国,35615.0,1
1,华东销售经理,销售经理,华东,25959.0,2
2,华南销售经理,销售经理,华南,25056.0,3
3,西南销售经理,销售经理,西南,21927.0,4
4,华北销售经理,销售经理,华北,18083.0,5
5,华中销售经理,销售经理,华中,18032.0,6
6,杨芳,销售代表,华北,14935.0,7
7,林杰,销售代表,华北,13872.0,8
8,马杰,销售代表,华中,12814.0,9
9,高刚,销售代表,华东,12575.0,10


In [9]:
# 4.2 递归 CTE 打印组织层级
q('''
WITH RECURSIVE org AS (
  SELECT employee_id, name, manager_id, 1 AS level, CAST(name AS TEXT) AS path
  FROM employees WHERE manager_id IS NULL
  UNION ALL
  SELECT e.employee_id, e.name, e.manager_id, o.level+1, o.path || ' > ' || e.name
  FROM employees e JOIN org o ON e.manager_id = o.employee_id
)
SELECT level, path FROM org ORDER BY path LIMIT 15
''')

,level,path
0,1,张总
1,2,张总 > 市场总监
2,3,张总 > 市场总监 > 华东市场经理
3,4,张总 > 市场总监 > 华东市场经理 > 陈伟
4,3,张总 > 市场总监 > 华中市场经理
5,4,张总 > 市场总监 > 华中市场经理 > 刘娟
6,3,张总 > 市场总监 > 华北市场经理
7,4,张总 > 市场总监 > 华北市场经理 > 孙涛
8,4,张总 > 市场总监 > 华北市场经理 > 杨娜
9,3,张总 > 市场总监 > 华南市场经理


In [10]:
# 4.3 各部门人数与平均薪资
q('''
SELECT department, COUNT(*) AS 人数, ROUND(AVG(salary),0) AS 平均薪资
FROM employees GROUP BY department ORDER BY 平均薪资 DESC
''')

,department,人数,平均薪资
0,管理层,1,60000.0
1,技术部,13,17357.0
2,销售部,14,17297.0
3,市场部,12,17281.0
4,运营部,14,17146.0


**结论 4**：组织是标准三级架构（总经理→总监→区域经理→专员）。技术部平均薪资最高，销售部人数最多。薪资层级清晰，没有明显倒挂。

## 模块 5 · 累计与同比环比

In [11]:
# 5.1 年度销售额 + 同比
q('''
WITH y AS (
  SELECT strftime('%Y', o.order_date) AS yr, SUM(oi.sales) AS sales
  FROM orders o JOIN order_items oi ON oi.order_id=o.order_id
  GROUP BY yr
)
SELECT yr, ROUND(sales,0) AS 销售额,
       ROUND((sales - LAG(sales) OVER (ORDER BY yr))
             *100.0 / LAG(sales) OVER (ORDER BY yr), 1) AS 同比_pct
FROM y ORDER BY yr
''')

,yr,销售额,同比_pct
0,2022,6044515.0,NaN
1,2023,5936208.0,-1.8
2,2024,6693231.0,12.8
3,2025,6366372.0,-4.9


In [12]:
# 5.2 年度累计销售额
q('''
WITH y AS (
  SELECT strftime('%Y', order_date) AS yr, SUM(oi.sales) AS sales
  FROM orders o JOIN order_items oi ON oi.order_id=o.order_id GROUP BY yr
)
SELECT yr, ROUND(sales,0) AS 年销售额,
       ROUND(SUM(sales) OVER (ORDER BY yr ROWS UNBOUNDED PRECEDING),0) AS 累计
FROM y ORDER BY yr
''')

,yr,年销售额,累计
0,2022,6044515.0,6044515.0
1,2023,5936208.0,11980723.0
2,2024,6693231.0,18673954.0
3,2025,6366372.0,25040327.0


In [13]:
# 5.3 最近一年月度环比（2025 年）
q('''
WITH m AS (
  SELECT strftime('%Y-%m', o.order_date) AS ym, SUM(oi.sales) AS sales
  FROM orders o JOIN order_items oi ON oi.order_id=o.order_id
  WHERE o.order_date >= '2025-01-01'
  GROUP BY ym
)
SELECT ym, ROUND(sales,0) AS 月销售额,
       ROUND((sales-LAG(sales) OVER (ORDER BY ym))*100.0
             /LAG(sales) OVER (ORDER BY ym),1) AS 环比_pct
FROM m ORDER BY ym
''')

,ym,月销售额,环比_pct
0,2025-01,563369.0,NaN
1,2025-02,351631.0,-37.6
2,2025-03,440918.0,25.4
3,2025-04,575943.0,30.6
4,2025-05,528285.0,-8.3
5,2025-06,673077.0,27.4
6,2025-07,513977.0,-23.6
7,2025-08,503829.0,-2.0
8,2025-09,479044.0,-4.9
9,2025-10,527431.0,10.1


**结论 5**：2022→2023 微降 1.8%，2024 大涨 12.8%，但 2025 又回落 4.9%。增长不是单边向上，2025 失速是个预警信号，需要结合渠道/促销找原因。

## 模块 6 · 组内 TopN 与异常发现

In [14]:
# 6.1 每地区销售额 Top2 城市
q('''
WITH c AS (
  SELECT o.region, o.city, SUM(oi.sales) AS sales
  FROM orders o JOIN order_items oi ON oi.order_id=o.order_id
  GROUP BY o.region, o.city
), r AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY region ORDER BY sales DESC) AS rn
  FROM c
)
SELECT region, city, ROUND(sales,0) AS 销售额, rn
FROM r WHERE rn <= 2 ORDER BY region, rn
''')

,region,city,销售额,rn
0,华东,上海,1702056.0,1
1,华东,宁波,1342969.0,2
2,华中,长沙,1817178.0,1
3,华中,郑州,1193826.0,2
4,华北,青岛,944607.0,1
5,华北,天津,791205.0,2
6,华南,佛山,1842927.0,1
7,华南,厦门,1660017.0,2
8,西南,成都,967566.0,1
9,西南,重庆,860116.0,2


In [15]:
# 6.2 负利润明细：多少行、亏多少
q('''
SELECT COUNT(*) AS 亏本明细行数, ROUND(SUM(profit),0) AS 亏本总额
FROM order_items WHERE profit < 0
''')

,亏本明细行数,亏本总额
0,823,-458577.0


In [16]:
# 6.3 按折扣分组，定位哪一档在亏钱
q('''
SELECT discount AS 折扣, COUNT(*) AS 行数,
       ROUND(SUM(profit),0) AS 总利润
FROM order_items GROUP BY discount ORDER BY discount
''')

,折扣,行数,总利润
0,0.0,2437,4858310.0
1,0.1,1601,2521850.0
2,0.2,1164,1101241.0
3,0.4,776,71464.0
4,0.6,429,-319557.0


**结论 6（关键异常）**：823 条明细在亏本，合计约 -45.9 万。按折扣拆开看：**6 折（discount=0.6）这一档单独亏了约 32 万**——这是“大促卖越多亏越多”的典型陷阱。建议立即复盘 6 折促销的成本结构，要么抬高折扣门槛，要么淘汰高成本商品。其余折扣档都健康盈利。

## 给 CEO 的 5 条结论（汇总）

1. **盘子健康**：4 年累计销售 2504 万、利润 823 万，整体利润率 32.9%。
2. **收入高度依赖科技产品**：占 69%；办公用品仅 1.6%，建议精简或捆绑。
3. **2025 增长失速**：同比 -4.9%，而 2024 曾 +12.8%，需排查渠道。
4. **6 折促销在亏钱**：823 行亏本明细，6 折档独亏 32 万，应立即调整促销策略。
5. **沉睡客户需召回**：平均 recency 155 天，老客户唤醒 ROI 高于拉新。

In [17]:
conn.close()

## 来源与延伸

- 本答案基于 `data/shop.db`（种子 20260913）。可把同样的分析迁移到 `data/superstore.db` 做对比。
- 窗口函数官方文档：https://www.sqlite.org/windowfunctions.html
- 递归 CTE：https://www.sqlite.org/lang_with.html
- 面试复盘：DataLemur https://datalemur.com/ ；LeetCode Database https://leetcode.com/problemset/database/